# Tutorial 3 — Ordinal grid landscape (continuous variables on a discrete grid)

**Dataset:** Wang *et al.* 2026 (Nat. Commun.) — W-Re-Os refractory multi-principal element alloy library.

**Why this is interesting for GraphFLA.** The three compositions are *physically continuous* atomic percentages, but the experiment samples them on a **3 at.% grid**. This is the common case in chemistry / materials / engineering: the inputs are numeric and **ordered** (a step of +3 at.% is conceptually closer to the current state than a step of +30 at.%), so the right declaration is `'ordinal'`. `'ordinal'` tells GraphFLA the values have a meaningful order — the encoded codes preserve that order (W = 3 → 0, W = 6 → 1, …, W = 94 → 31) so any downstream analysis that walks the grid sees a consistent rank.

**Note on the neighbour structure.** GraphFLA's default Hamming-1 neighbourhood for both `'ordinal'` and `'categorical'` axes connects two configurations whenever they differ on exactly one axis (any value change). The `'ordinal'` declaration is therefore primarily about *encoding semantics* — it does not, in the current default, restrict neighbours to ±1 grid step. If you need a strict grid-step neighbourhood, post-filter the edge list on the absolute difference of the underlying numeric values.

**Search space.** The library obeys a simplex constraint `W + Re + Os = 100`. Two of the three columns are independent; the third is determined. We build the landscape over `(W, Re)` only — adding `Os` would create no extra information and would silently bias the Hamming-1 neighbourhood (a step in `W` *requires* a compensating step in `Re` or `Os`, so no two configurations would ever differ in exactly one slot if all three are independent ordinal axes).

Fitness is the paper's composite TOPSIS score `Ci ∈ [0, 1]` (higher = better all-round refractory alloy).

In [ ]:
import pandas as pd
from graphfla.landscape import Landscape
from graphfla.analysis import (
    lo_ratio, autocorrelation, fitness_distance_corr,
    classify_epistasis, gradient_intensity,
)

df = pd.read_csv('../data/Materials/WReOs/simplex.csv')
print(df.shape, '— phase counts:')
print(df['phase_label'].value_counts())
df.head()

## 1. Build the landscape

In [ ]:
X = df[['W', 'Re']]
f = df['TOPSIS_Ci']

ls = Landscape(maximize=True)
ls.build_from_data(
    X, f,
    data_types={'W': 'ordinal', 'Re': 'ordinal'},
    verbose=False,
)
print(f'nodes: {ls.graph.vcount()}, edges: {ls.graph.ecount()}')

## 2. Topographical features

These should be in the same range as the numbers in the project's `DATASETS.md` landscape-metrics table (`WReOs` row). Small differences vs. that table come from the random-walk sampling used by `autocorrelation` and from `classify_epistasis(approximate=True)`.

In [ ]:
print(f'lo_ratio          : {lo_ratio(ls):.4f}')
print(f'autocorr (lag1)   : {autocorrelation(ls):.4f}')
print(f'fdc (Spearman)    : {fitness_distance_corr(ls):.4f}')
print(f'gradient_intensity: {gradient_intensity(ls):.4f}')

print('\nEpistasis (approximate; 4-node squares):')
for k, v in classify_epistasis(ls, approximate=True, sample_cut_prob=0.5).items():
    print(f'  {k:>26}: {v:.4f}')

## 3. Visualise the grid

Because the search space is 2-D, we can render the fitness surface directly.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(df['W'], df['Re'], c=df['TOPSIS_Ci'], cmap='viridis', s=12)
ax.set_xlabel('W (at. %)'); ax.set_ylabel('Re (at. %)')
ax.set_title(f'W-Re-Os TOPSIS Ci ({len(df)} alloys on 3 at.% grid)')
plt.colorbar(sc, ax=ax, label='TOPSIS Ci')
plt.tight_layout(); plt.show()

## Takeaways

- Use `'ordinal'` whenever a numeric axis has a meaningful order *and* you want the encoded code to reflect that order (useful for any downstream analysis or visual that walks the grid).
- If a numeric axis has irregular spacing, you can still declare it ordinal — GraphFLA only uses the rank order, not the absolute distance.
- If a variable is constrained (like the W+Re+Os = 100 simplex here), drop one redundant axis before building. The Hamming-1 graph assumes axes can be perturbed independently.
- The same recipe applies to: any compositional grid, temperature scans, pH scans, concentration scans, time-course inputs.